# AlloyGEN: generative inverse design of alloys

**The normal way to use ML for materials** is *forward*: given a composition, predict a property.
**Inverse design flips it**: given a target property, generate a composition that should have it.
That is what the Materials 4.0 "AlloyGEN" project does, developing new wear-resistant alloys with
*diffusion-based, structure-aware generative AI*.

This notebook builds a small, honest, working version of that idea on **real data**, so you can see
the whole pipeline end to end. It follows the approach in the recent literature (a VAE that learns a
latent space of alloys, then generation in that space conditioned on a target property, e.g.
[Abu-Mualla et al., 2026](https://advanced.onlinelibrary.wiley.com/doi/10.1002/aidi.202500069)).

**The pipeline:**

```
real alloys ->  [ surrogate ]  composition -> strength  (forward model, honest CV)
            ->  [ CVAE ]        learns to GENERATE alloys conditioned on a target strength
            ->  [ screen ]      generate for a high target, validate with the surrogate,
                                keep the novel, plausible ones
```

**Honest scope up front:** the real project targets wear resistance; here I use **yield strength**
of steels (a real, measured mechanical property) as an accessible stand-in. The *method* is
identical, and I am explicit about the limits at the end. Everything runs on real data, nothing is
simulated.

## Step 0 - Kernel

Use the Python that has **torch, matminer, pymatgen, and scikit-learn** (your `C:\Python314`).
Select it in the kernel picker before running.

In [ ]:
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from matminer.datasets import load_dataset
from pymatgen.core import Composition
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from scipy.spatial.distance import cdist

torch.manual_seed(0); np.random.seed(0)   # reproducible
print("ready | torch", torch.__version__)

## Step 1 - Real alloy data

`matbench_steels` is a benchmark set of **312 real steels**, each a measured composition and its
**yield strength** (MPa). We turn every composition string into a fixed-length vector of **element
fractions** (Fe, C, Cr, Ni, ...), the numeric form both the surrogate and the generator work with.

In [ ]:
df = load_dataset("matbench_steels")
print("rows:", len(df))
print(df.head(3).to_string())

# turn each composition into a vector of element fractions (they sum to 1)
comps = [Composition(c) for c in df["composition"]]
elements = sorted({str(e) for c in comps for e in c.elements})   # the 14 elements in the dataset
def to_vec(c):
    d = c.get_el_amt_dict(); total = sum(d.values())
    return np.array([d.get(el, 0.0) / total for el in elements])
X = np.array([to_vec(c) for c in comps])                          # (312, 14) fraction vectors
y = df["yield strength"].values.astype(float)                    # (312,) MPa
print("\nfeature matrix:", X.shape, "| elements:", elements)
print("strength range: %.0f to %.0f MPa" % (y.min(), y.max()))

## Step 2 - Forward surrogate: composition to strength

Before we can *design*, we need a model that scores a candidate. A random forest maps the element
fractions to strength. We check it honestly with 5-fold cross-validation (train and test on
different alloys), then fit it on all the data to use as our screening judge later.

In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=0)
r2 = cross_val_score(rf, X, y, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
print("surrogate 5-fold CV R2: %.3f" % r2)   # honest out-of-sample skill
rf.fit(X, y)                                  # now fit on all data, to judge generated alloys

## Step 3 - The generative model: a conditional VAE

A **variational autoencoder (VAE)** learns to compress alloys into a small "latent space" and
decode them back. Making it **conditional** on the target strength is the trick: we feed the
strength in during training, so at generation time we can *ask* for a strength and get an alloy
shaped for it.

Two design choices worth noting in the code:
- the decoder ends in a **softmax**, which guarantees every generated alloy is a **valid composition**
  (non-negative fractions that sum to 1),
- the loss is **reconstruction + a small KL term**, the standard VAE objective that keeps the latent
  space smooth so we can sample from it.

In [ ]:
Xt = torch.tensor(X, dtype=torch.float32)
ymean, ystd = y.mean(), y.std()
cond = torch.tensor((y - ymean) / ystd, dtype=torch.float32).unsqueeze(1)   # normalised target
D, L = X.shape[1], 8                                                        # input dim, latent dim

class CVAE(nn.Module):
    def __init__(self):
        super().__init__()
        # encoder sees the alloy AND its strength -> latent distribution (mu, log-var)
        self.enc = nn.Sequential(nn.Linear(D + 1, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.mu, self.lv = nn.Linear(32, L), nn.Linear(32, L)
        # decoder rebuilds the alloy from a latent point AND the target strength
        self.dec = nn.Sequential(nn.Linear(L + 1, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, D))
    def encode(self, x, c):
        h = self.enc(torch.cat([x, c], 1)); return self.mu(h), self.lv(h)
    def decode(self, z, c):
        return torch.softmax(self.dec(torch.cat([z, c], 1)), dim=-1)   # -> valid composition
    def forward(self, x, c):
        mu, lv = self.encode(x, c)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(lv)            # reparameterisation trick
        return self.decode(z, c), mu, lv

model = CVAE(); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(4000):
    opt.zero_grad()
    recon, mu, lv = model(Xt, cond)
    rec_loss = nn.functional.mse_loss(recon, Xt, reduction="sum") / len(Xt)   # rebuild the alloy
    kl = -0.5 * torch.mean(torch.sum(1 + lv - mu.pow(2) - lv.exp(), dim=1))   # keep latent smooth
    (rec_loss + 0.02 * kl).backward(); opt.step()
    if epoch % 1000 == 0: print("epoch %4d  recon %.4f" % (epoch, float(rec_loss)))
print("CVAE trained.")

## Step 4 - The key test: does the conditioning actually work?

This is the experiment that proves the generator is real, not decoration. We ask it to generate
alloys for a **low, medium, and high target strength**, then score each batch with the independent
surrogate. If the model genuinely learned the property, the surrogate-rated strength of the
generated alloys should **rise as we raise the target**. It does.

In [ ]:
targets_pct = [25, 50, 75, 95]
gen_means = []
for pct in targets_pct:
    tc = (np.percentile(y, pct) - ymean) / ystd
    with torch.no_grad():
        gen = model.decode(torch.randn(2000, L), torch.full((2000, 1), float(tc))).numpy()
    gen_means.append(rf.predict(gen).mean())
    print("target p%-2d (%4.0f MPa)  ->  generated alloys avg %.0f MPa" %
          (pct, np.percentile(y, pct), gen_means[-1]))

plt.figure(figsize=(5, 4))
plt.plot([np.percentile(y, p) for p in targets_pct], gen_means, "o-")
plt.xlabel("target strength asked for (MPa)"); plt.ylabel("avg strength of generated alloys (MPa)")
plt.title("The generator responds to the target"); plt.tight_layout(); plt.show()

## Step 5 - Inverse design: propose novel high-strength alloys

Now the payoff. We generate a big batch aimed at a **high** target, screen them with the surrogate,
and keep only candidates that are **novel** (chemically different from every training alloy) and
**plausible** (iron-dominant, so they are actually steels). The top ones come out looking like
**maraging steels** (Fe-Ni-Co-Mo-Ti), a real high-strength family, which the model discovered on
its own from being asked for strength.

In [ ]:
tc = (np.percentile(y, 95) - ymean) / ystd
with torch.no_grad():
    cand = model.decode(torch.randn(4000, L), torch.full((4000, 1), float(tc))).numpy()
pred = rf.predict(cand)                       # surrogate score
novelty = cdist(cand, X).min(axis=1)          # distance to the nearest real alloy
fe = elements.index("Fe")
keep = (novelty > 0.03) & (cand.argmax(1) == fe)          # novel AND iron-dominant (a steel)
top = np.where(keep)[0][np.argsort(-pred[np.where(keep)[0]])][:5]

print("Top novel, plausible alloys the generator proposes for high strength:\n")
for i in top:
    alloy = {elements[j]: round(float(cand[i][j]), 3) for j in range(D) if cand[i][j] > 0.01}
    print("  %.0f MPa | novelty %.3f | %s" % (pred[i], novelty[i], alloy))

## Step 6 - Honest limits, and how the real AlloyGEN goes further

This is a working proof of concept, and being clear about its edges is part of doing it well:

- **The surrogate does not extrapolate.** A random forest cannot predict much beyond the strengths
  it was trained on, so the generated alloys cluster below the very top of the range even when we
  ask for more. The *trend* is right; the absolute ceiling is the surrogate's, not the generator's.
- **Small data.** 312 alloys is tiny for generative modelling. The paper this follows uses ~1,860.
- **Strength is a proxy** for the wear resistance the real project targets, and a real programme
  would validate candidates by **synthesis and testing**, not just a surrogate.

**How the full AlloyGEN (Materials 4.0) goes further:** a **latent diffusion** model instead of a
plain VAE (better distribution coverage), **structure-aware** features rather than composition
alone, far more data, and a closed loop with **experiment**. The pipeline shape, though, is exactly
this one.

**What this demonstrates:** the complete generative-inverse-design workflow, built and verified on
real data, forward surrogate, conditional generative model, property-targeted generation, and
honest validation. That is the AlloyGEN project in miniature, and the skill set it asks for.